# 🎙️ استوديو «رِواق» الآلي بضغطة زر واحدة (بدون واجهات معقدة ولا رفع ملفات!)

### طريقة الاستخدام في خطوتين فقط:
1. تأكد من القائمة العلوية أن نوع المعالج **T4 GPU**: (`Runtime` ➔ `Change runtime type` ➔ `T4 GPU`).
2. اضغط من القائمة العلوية: **`Runtime` ➔ `Run all`** (أو اضغط زر التشغيل ▶ على الخلية رقم **1** ثم الخلية رقم **2**) — وسيقوم النظام **تلقائياً** بقراءة عيناتك من مجلد **`رواق / عينة الصوت`** وقراءة النص من **`رواق / 06-VOICE-CLONING-STUDIO / سكريبت_الحلقة.txt`** وحفظ الصوت النهائي الماستر (`48kHz / -16 LUFS`) داخل **`رواق / 06-VOICE-CLONING-STUDIO / outputs`** في Google Drive مع عرض مشغل صوت مباشر هنا!

In [ ]:
# @title 1️⃣ الخلية الأولى: ربط Google Drive وتجهيز المحركات الصوتية تلقائياً
import os, subprocess, sys
from google.colab import drive

print('🔗 جاري ربط مساحة Google Drive (100GB)...')
drive.mount('/content/drive')

def sh(cmd):
    print(f'▶ {cmd}')
    subprocess.run(cmd, shell=True, check=True)

sh('apt-get update -qq && apt-get install -y -qq ffmpeg git')

REPO_DIR = '/content/my-voice-cloning'
BRANCH = 'arena/a44deae5-my-voice-cloning'
if not os.path.exists(REPO_DIR):
    sh(f'git clone -b {BRANCH} https://github.com/omarretco27/my-voice-cloning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} fetch origin {BRANCH} && git -C {REPO_DIR} reset --hard origin/{BRANCH}')

if not os.path.exists('/content/Habibi-TTS'):
    sh('git clone --depth 1 https://github.com/SWivid/Habibi-TTS.git /content/Habibi-TTS')
if not os.path.exists('/content/OpenVoice'):
    sh('git clone --depth 1 https://github.com/myshell-ai/OpenVoice.git /content/OpenVoice')

sh('pip install -q --upgrade pip wheel setuptools')
sh('pip install -q audioop-lts "f5-tts>=1.1.5" faster-whisper==1.2.1 edge-tts==7.3.1 soundfile scipy noisereduce librosa huggingface_hub safetensors hydra-core omegaconf cached_path tomli eng_to_ipa inflect unidecode pypinyin cn2an jieba')

import torch
print('\n✅ اكتمل التجهيز بنجاح! حالة كرت الشاشة GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠️ CPU فقط — يُفضّل تفعيل T4 GPU')

In [ ]:
# @title 2️⃣ الخلية الثانية: توليد الصوت الماستر تلقائياً من «عينة الصوت» و«سكريبت_الحلقة.txt» 🎧
import importlib, subprocess, sys

# تحديث تلقائي لآخر إصلاحات الكود في كل تشغيلة وتحديث الموديولات في الذاكرة
subprocess.run('git -C /content/my-voice-cloning fetch origin arena/a44deae5-my-voice-cloning && git -C /content/my-voice-cloning reset --hard origin/arena/a44deae5-my-voice-cloning', shell=True, check=True)

for p in ('/content/my-voice-cloning/colab', '/content/my-voice-cloning/scripts', '/content/Habibi-TTS/src', '/content/OpenVoice'):
    if p not in sys.path:
        sys.path.insert(0, p)

import prepare_dataset, voice_finish, transcribe_to_meta, habibi_egy_pipeline, auto_run_colab
for mod in (prepare_dataset, voice_finish, transcribe_to_meta, habibi_egy_pipeline, auto_run_colab):
    importlib.reload(mod)

result = auto_run_colab.run_zero_ui_pipeline(drive_root='/content/drive/MyDrive', episode_id='EP_001')

---
### 🌟 (اختياري) الخلية الثالثة: تدريب موديل استنساخ عميق دائم (`RVC v2 · 48kHz RMVPE`) على كل عيناتك في «عينة الصوت»
كلما أضفت تسجيلات أكثر بصوتك داخل مجلد **`رواق / عينة الصوت`** (يُنصح بـ 10 إلى 20 دقيقة لتحقيق أعلى تطابق إذاعي)، يمكنك تشغيل الخلية التالية مرة واحدة فقط ليحفظ الموديل الدائم على مساحة الـ 100 جيجا في Google Drive، وبعدها ستقوم الخلية رقم (2) تلقائياً بإنتاج النسخة الثالثة بهذا الموديل أيضاً!

In [ ]:
# @title 3️⃣ (اختياري) ضع علامة (✓) هنا فقط عندما تريد تدريب موديل RVC v2 الكامل على كل ملفات «عينة الصوت»
TRAIN_FULL_RVC_MODEL = False # @param {type:"boolean"}
EPOCHS = 200 # @param {type:"slider", min:100, max:400, step:25}

if not TRAIN_FULL_RVC_MODEL:
    print('ℹ️ تم تخطي تدريب RVC الطويل (لأن الخيار غير مفعّل) — ملفاتك الجاهزة موجودة الآن بالأعلى وفي Google Drive داخل outputs/!')
else:
    import os, subprocess, sys
    if not os.path.exists('/content/Applio'):
        subprocess.run('git clone --depth 1 https://github.com/IAHispano/Applio.git /content/Applio', shell=True, check=True)
        subprocess.run('pip install -q faiss-cpu torchcrepe praat-parselmouth pyworld peft pedalboard soxr onnxruntime-gpu', shell=True, check=True)
    for p in ('/content/my-voice-cloning/colab', '/content/my-voice-cloning/scripts'):
        if p not in sys.path:
            sys.path.insert(0, p)
    import auto_run_colab
    auto_run_colab.train_rvc_v2_one_click(drive_root='/content/drive/MyDrive', applio_dir='/content/Applio', epochs=int(EPOCHS))